# 第 25 课｜梯度累积与激活检查点
## 在有限显存下，保持正确的训练目标

**学习路径：**第 24 课 AMP → token 加权梯度累积 → 尾窗口 → 激活重计算 → Dropout/RNG → 三路训练 → 内存与时间口径。

本讲义和同名 `.py` 脚本分别独立可运行，不需要其他课程文件、检查点、外部语料或网络。只依赖 **PyTorch 与 Python 标准库**；继续使用已有环境。默认使用 CPU、FP32、单线程。

**学习目标**：能解释并实现为什么累积窗口只更新一次；能给变长文本写对损失分母；能证明重计算没有改变梯度；能区分激活载荷与实际峰值显存。

先看解释与短实验，再读模型和训练参考实现。标注“参考实现”的长单元默认折叠，可以展开；独立练习位于后半部分，答案在练习之后。**运行全部单元不代表完成练习。**

这是合成 token 数据的训练工程实验，不是中文、推理、编程能力评测。当前提供的实际执行输出来自 CPU；CUDA 路径需在你的机器实测。

## 0. 三种“节省”不要混在一起

| 方法 | 主要改变什么 | 不直接解决什么 |
|---|---|---|
| 梯度累积 | 每次只建立一个小批次的前向/反向图；跨小批次保留梯度 | 参数、参数梯度和优化器状态仍需存放 |
| 激活检查点 | 少保存区域内部的中间值，反向时重新计算 | 不把模型参数压缩，不自动提供断点续训 |
| 混合精度 | 一部分算子以 FP16/BF16 执行 | 不保证整个训练内存减半 |

**Activation checkpointing ≠ 磁盘 checkpoint。** 本课前者是激活重计算；`torch.save` 保存模型则是另一个问题。[PyTorch 检查点文档](https://docs.pytorch.org/docs/2.10/checkpoint.html)解释了它以额外计算换取较少保留激活的机制。

第 19–22 课的 **KV cache 服务于增量推理**，本课训练前向不更新 KV cache。不要把会追加历史的有状态函数放进需要重复执行的 checkpoint 区域。

In [1]:
import argparse
from contextlib import contextmanager
import copy
from dataclasses import asdict, dataclass, replace
import gc
import hashlib
import json
import math
from pathlib import Path
import platform
import statistics
import time
from typing import Iterable, Iterator

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint

SOURCES = {
    "checkpoint": "https://docs.pytorch.org/docs/2.10/checkpoint.html",
    "amp_accumulation": "https://docs.pytorch.org/docs/2.10/notes/amp_examples.html",
    "checkpoint_tradeoff": "https://pytorch.org/blog/activation-checkpointing-techniques/",
    "saved_tensors": "https://docs.pytorch.org/docs/stable/notes/autograd.html",
    "cuda_peak": "https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html",
    "cross_entropy": "https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html",
    "numerical_accuracy": "https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html",
}
PAD, BOS, EOS, IGNORE = 0, 1, 2, -100
PRECISIONS = ("fp32", "amp_bf16", "amp_fp16")


def positive_int(n, name):
    if type(n) is not int or n <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受 bool")


def choose_device(name):
    if name not in ("cpu", "cuda", "auto"):
        raise ValueError("device 必须是 cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；不会静默回退。请检查 PyTorch 安装或指定 cpu。")
    return torch.device(name)


def autocast_context(precision, device):
    if precision not in PRECISIONS:
        raise ValueError(f"precision 必须属于 {PRECISIONS}")
    if precision == "fp32":
        return torch.autocast(device_type=device.type, enabled=False)
    dtype = torch.bfloat16 if precision == "amp_bf16" else torch.float16
    return torch.autocast(device_type=device.type, dtype=dtype)


def make_scaler(precision, device):
    return torch.amp.GradScaler(device.type, enabled=precision == "amp_fp16",
                               init_scale=1024.0, growth_interval=200)


@contextmanager
def reference_precision():
    """局部固定 FP32 matmul 参考策略；退出时恢复，不混用新旧 API。"""
    matmul = torch.backends.cuda.matmul
    if hasattr(matmul, "fp32_precision"):
        old = matmul.fp32_precision
        matmul.fp32_precision = "ieee"
        try:
            yield
        finally:
            matmul.fp32_precision = old
    else:
        old = torch.get_float32_matmul_precision()
        torch.set_float32_matmul_precision("highest")
        try:
            yield
        finally:
            torch.set_float32_matmul_precision(old)


def fingerprint(state):
    h = hashlib.sha256()
    for name, value in sorted(state.items()):
        h.update(name.encode())
        h.update(str(value.dtype).encode())
        h.update(str(tuple(value.shape)).encode())
        h.update(bytes(value.detach().cpu().contiguous().view(torch.uint8).flatten().tolist()))
    return h.hexdigest()


@dataclass(frozen=True)
class Config:
    vocab_size: int = 16
    d_model: int = 48
    n_heads: int = 4
    n_kv_heads: int = 2
    n_layers: int = 2
    ff_dim: int = 128
    max_len: int = 32
    dropout: float = 0.0
    eps: float = 1e-6

    def __post_init__(self):
        for name in ("vocab_size", "d_model", "n_heads", "n_kv_heads", "n_layers", "ff_dim", "max_len"):
            positive_int(getattr(self, name), name)
        if self.vocab_size < 16 or self.max_len < 9:
            raise ValueError("本课数据要求 vocab_size>=16，max_len>=9")
        if self.d_model % self.n_heads or self.n_heads % self.n_kv_heads:
            raise ValueError("D 必须被 Hq 整除；Hq 必须被 Hkv 整除")
        if (self.d_model // self.n_heads) % 2:
            raise ValueError("RoPE 的每头维度必须为偶数")
        if not (0 <= self.dropout < 1) or not math.isfinite(self.eps) or self.eps <= 0:
            raise ValueError("dropout/eps 非法")

In [2]:
# 本 Notebook 的唯一实验控制区；不需要修改模型实现。
DEVICE = "cpu"             # CUDA 配置好后可改为 "cuda"；"auto" 会报告实际设备
PRECISION = "fp32"         # 可选 "amp_bf16"；"amp_fp16" 在本课只用于 CUDA
EPOCHS = 8
STRATEGIES = ("full", "accum", "checkpoint")
OUTPUT_DIR = "lesson_25_outputs"
RUN_RESOURCE_BENCHMARK = False  # 额外耗时/显存测试；默认先完成正确性和训练
BENCH_LENGTH = 64

torch.set_num_threads(1)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
print("实验实际设备:", choose_device(DEVICE))
print("报告目录:", Path(OUTPUT_DIR).resolve())

Python: 3.13.5
PyTorch: 2.10.0+cpu
CUDA 可用: False
实验实际设备: cpu
报告目录: /mnt/data/lesson_25_validation/notebook_run/lesson_25_outputs


## 1. 梯度为什么能累积？

对于互不耦合的样本、同一组参数和可加的损失：

\[
L(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell_i(\theta),\qquad
\nabla L=\frac1N\sum_i\nabla\ell_i.
\]

所以，可以拆成多个 micro-batch，分别计算对应的梯度贡献，再做**一次**优化器更新。反向传播时 `.grad` 默认相加，正好可实现这个过程。

例如，每次放 3 条、累积 4 次，一般得到 12 条样本的有效批次；但最后的不足批次要按实际数量处理。变长文本还需要计算整个窗口的**有效 token 数**。

“micro-batch”是一次前向/反向的数据量；“累积窗口”是共同产生一次参数更新的一组小批次。本课是单设备，不实现 DDP；不要直接把这里的归一化搬到多卡平均梯度代码里而不重新推导。

In [3]:
# 对比一次性 backward 与两次 backward；两次之间不能更新参数。
w_full = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
(((w_full - 1)**2 + (w_full - 3)**2) / 2).backward()

w_micro = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
((w_micro - 1)**2 / 2).backward()
print("第一个 micro 之后的梯度:", w_micro.grad.item())
((w_micro - 3)**2 / 2).backward()
print("第二个 micro 之后的梯度:", w_micro.grad.item())
print("一次性计算的梯度:", w_full.grad.item())
torch.testing.assert_close(w_full.grad, w_micro.grad)

第一个 micro 之后的梯度: -1.0
第二个 micro 之后的梯度: -4.0
一次性计算的梯度: -4.0


### 1.1 等价需要什么条件？

我们首先关闭 Dropout，使用没有 BatchNorm、没有跨样本对比损失的逐序列解码器。每个小批次使用同一组参数；所有小批次反向完成后才裁剪和更新。这样才适合验证数学上的大批次等价。

有限精度、矩阵形状和后端会影响浮点加法顺序，不能要求所有路径逐位相等。开启 Dropout 后，拆批与整批的随机数消费方式也可能不同，即使设置相同种子，也不能自动要求两种批划分逐位一致。[数值精度说明](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html)

**梯度累积本身不会扩大模型单条序列可处理的长度。** 如果一条长序列的激活就放不下，仍可能需要更短序列、检查点、优化注意力内核等方法。

## 2. 变长文本：不能机械地 `loss / accumulation_steps`

本课采用 **每个有效 token 等权** 的目标。设第 \(j\) 个 micro-batch 有 \(n_j\) 个有效目标，其损失和为 \(S_j\)。正确的窗口损失为：

\[
L=\frac{\sum_jS_j}{\sum_jn_j}.
\]

因此每个 micro-batch 应反传：

\[
L_j=\frac{S_j}{N_{window}},\quad N_{window}=\sum_jn_j.
\]

若一个 micro 有 9 个目标，均值为 1，另一个只有 1 个目标，均值为 9：正确结果是 **1.8**，两个均值再平均却是 **5**。

只有各小批次有效目标数相等、最后窗口也确实有相同份数等条件成立时，`micro_mean / K` 才对应这里的目标。其他按样本等权的任务可能有不同定义；重要的是先确定目标，再推导分母。

In [4]:
def make_data():
    """不同 (a,b) 对只出现一次；重复次数造成长度差异。不是语言能力数据集。"""
    rows = [[BOS] + [a, b]*(1+(a+2*b) % 4) + [EOS]
            for a in range(4, 16) for b in range(4, 16) if a != b]
    order = torch.randperm(len(rows), generator=torch.Generator().manual_seed(2502)).tolist()
    selected = [rows[i] for i in order[:60]]
    return selected[:43], selected[43:]


def collate(rows):
    if not rows or any(len(row) < 2 for row in rows):
        raise ValueError("需要非空样本，且每条至少包含两个 token")
    xs = [torch.tensor(row[:-1], dtype=torch.long) for row in rows]
    ys = [torch.tensor(row[1:], dtype=torch.long) for row in rows]
    x = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=PAD)
    y = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=IGNORE)
    return x, y


def iter_microbatches(rows, micro_size, order=None):
    positive_int(micro_size, "micro_size")
    if order is None:
        order = list(range(len(rows)))
    for start in range(0, len(order), micro_size):
        yield collate([rows[i] for i in order[start:start+micro_size]])


def accumulation_windows(batches: Iterable, accumulation_steps: int) -> Iterator[list]:
    """只暂存当前窗口的 CPU batch，不把整个窗口的图或 GPU 激活存起来。"""
    positive_int(accumulation_steps, "accumulation_steps")
    window = []
    for batch in batches:
        window.append(batch)
        if len(window) == accumulation_steps:
            yield window
            window = []
    if window:  # 必须执行不足 K 个 micro-batch 的最后一个窗口
        yield window


def valid_tokens(targets):
    if targets.dtype != torch.long or targets.ndim != 2:
        raise ValueError("targets 必须是 long [B,T]")
    return int((targets != IGNORE).sum())


def loss_sum(logits, targets):
    if targets.dtype != torch.long or targets.shape != logits.shape[:-1]:
        raise ValueError("logits/targets 形状或类型不匹配")
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                           ignore_index=IGNORE, reduction="sum")


def denominator_demo():
    # 9 个有效目标的 micro-batch 均值为 1；另一个只有 1 个目标，均值为 9。
    return {"counts": [9, 1], "means": [1.0, 9.0],
            "correct_token_mean": (9*1.0+1*9.0)/10,
            "wrong_mean_of_means": (1.0+9.0)/2}


def window_summary(rows, micro_size=3, accumulation_steps=4):
    result = []
    for win in accumulation_windows(iter_microbatches(rows, micro_size), accumulation_steps):
        result.append({"microbatches": len(win), "samples": sum(x.shape[0] for x, _ in win),
                       "tokens": sum(valid_tokens(y) for _, y in win)})
    return result

In [5]:
print(json.dumps(denominator_demo(), ensure_ascii=False, indent=2))
TRAIN_ROWS, VAL_ROWS = make_data()
print("训练条数 / 验证条数:", len(TRAIN_ROWS), len(VAL_ROWS))
print("训练完整序列长度:", sorted(set(map(len, TRAIN_ROWS))))
example_x, example_y = collate(TRAIN_ROWS[:3])
print("输入（0 是 PAD）:"); print(example_x)
print("标签（-100 被忽略）:"); print(example_y)
print("有效目标数量:", valid_tokens(example_y))

{
  "counts": [
    9,
    1
  ],
  "means": [
    1.0,
    9.0
  ],
  "correct_token_mean": 1.8,
  "wrong_mean_of_means": 5.0
}
训练条数 / 验证条数: 43 17
训练完整序列长度: [4, 6, 8, 10]
输入（0 是 PAD）:
tensor([[ 1, 14,  8, 14,  8, 14,  8,  0,  0],
        [ 1, 15,  4, 15,  4, 15,  4, 15,  4],
        [ 1,  9, 15,  9, 15,  9, 15,  9, 15]])
标签（-100 被忽略）:
tensor([[  14,    8,   14,    8,   14,    8,    2, -100, -100],
        [  15,    4,   15,    4,   15,    4,   15,    4,    2],
        [   9,   15,    9,   15,    9,   15,    9,   15,    2]])
有效目标数量: 25


### 2.1 尾窗口不能丢，也不能假装它是满的

43 条训练文本，每次 3 条，会得到 15 个 micro-batch：最后一个只有 1 条。每 4 个 micro-batch 更新一次，窗口大小为：

```text
micro-batch 数：4、4、4、3
样本数：      12、12、12、7
```

最后窗口应处理真实的 7 条样本，并用它们实际的有效 token 总数做分母。不应该直接丢弃，也不应该仍除以 12 条或 4 个均值。

预先统计分母只需要读取 CPU 标签，不需要把整个窗口的数据提前送到 GPU，更不需要提前计算所有前向图。

In [6]:
windows = window_summary(TRAIN_ROWS, micro_size=3, accumulation_steps=4)
for i, item in enumerate(windows, start=1):
    print(f"更新窗口 {i}:", item)
print("本轮更新次数:", len(windows))
print("本轮处理样本数:", sum(item["samples"] for item in windows))
assert sum(item["samples"] for item in windows) == 43

更新窗口 1: {'microbatches': 4, 'samples': 12, 'tokens': 84}
更新窗口 2: {'microbatches': 4, 'samples': 12, 'tokens': 78}
更新窗口 3: {'microbatches': 4, 'samples': 12, 'tokens': 56}
更新窗口 4: {'microbatches': 3, 'samples': 7, 'tokens': 49}
本轮更新次数: 4
本轮处理样本数: 43


## 3. 模型参考：加入检查点，不更换已经学过的结构

仍然使用 RoPE、GQA、RMSNorm、SwiGLU、Pre-Norm 残差和词表输出头。默认两层、隐藏维度 48、4 个 Query 头、2 个 KV 头，**52,464 个参数**。

下方三个参考单元与脚本一致，不需要逐行重写旧组件。重点看 `TinyDecoder.forward` 内的 **block 循环**。参数保留 FP32；数学对照会临时使用 FP64。

In [7]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
            z = x.to(dtype)
            z = z * torch.rsqrt(z.square().mean(-1, keepdim=True) + self.eps)
            z = z * self.weight.to(dtype)
        return z.to(x.dtype)


def rope(x, inv_freq):
    with torch.autocast(device_type=x.device.type, enabled=False):
        dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
        pos = torch.arange(x.shape[-2], device=x.device, dtype=dtype)
        angle = pos[:, None] * inv_freq.to(dtype)[None, :]
        c, s = angle.cos()[None, None], angle.sin()[None, None]
        z = x.to(dtype)
        a, b = z[..., 0::2], z[..., 1::2]
        out = torch.stack((a*c-b*s, a*s+b*c), -1).flatten(-2)
    return out.to(x.dtype)

In [8]:
class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.d = cfg, cfg.d_model // cfg.n_heads
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.d, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.d, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.register_buffer("inv_freq", 10000.0 ** (-torch.arange(0, self.d, 2).float()/self.d))

    def forward(self, x, allowed):
        b, t, _ = x.shape
        q = self.q(x).reshape(b, t, self.cfg.n_heads, self.d).transpose(1, 2)
        k = self.k(x).reshape(b, t, self.cfg.n_kv_heads, self.d).transpose(1, 2)
        v = self.v(x).reshape(b, t, self.cfg.n_kv_heads, self.d).transpose(1, 2)
        q, k = rope(q, self.inv_freq), rope(k, self.inv_freq)
        # 显式 mask：True 允许读取；pad query 保留可访问 BOS，避免全屏蔽行。
        z = F.scaled_dot_product_attention(
            q, k, v, attn_mask=allowed, dropout_p=0.0,
            enable_gqa=self.cfg.n_heads != self.cfg.n_kv_heads,
        )
        return self.out(z.transpose(1, 2).reshape(b, t, -1))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg.d_model, cfg.eps), RMSNorm(cfg.d_model, cfg.eps)
        self.attn = Attention(cfg)
        self.gate = nn.Linear(cfg.d_model, cfg.ff_dim, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.ff_dim, bias=False)
        self.down = nn.Linear(cfg.ff_dim, cfg.d_model, bias=False)
        self.drop1, self.drop2 = nn.Dropout(cfg.dropout), nn.Dropout(cfg.dropout)

    def forward(self, x, allowed):
        x = x + self.drop1(self.attn(self.norm1(x), allowed))
        h = self.norm2(x)
        return x + self.drop2(self.down(F.silu(self.gate(h)) * self.up(h)))

In [9]:
class TinyDecoder(nn.Module):
    def __init__(self, cfg=Config(), activation_checkpointing=False, preserve_rng_state=True):
        super().__init__()
        self.cfg = cfg
        self.activation_checkpointing = activation_checkpointing
        self.preserve_rng_state = preserve_rng_state
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.eps)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, ids):
        if ids.dtype != torch.long or ids.ndim != 2 or min(ids.shape) == 0:
            raise ValueError("ids 必须是非空 long [B,T]")
        if ids.shape[1] > self.cfg.max_len:
            raise ValueError("超过最大长度")
        if bool((ids < 0).any()) or bool((ids >= self.cfg.vocab_size).any()):
            raise ValueError("token 编号越界")
        if bool((ids[:, 0] == PAD).any()) or bool(((ids[:, :-1] == PAD) & (ids[:, 1:] != PAD)).any()):
            raise ValueError("本课只支持非空、右侧补齐的序列")
        t = ids.shape[1]
        causal = torch.ones(t, t, device=ids.device, dtype=torch.bool).tril()
        allowed = causal[None, None] & (ids != PAD)[:, None, None, :]
        x = self.embedding(ids)
        for block in self.blocks:
            if self.activation_checkpointing and self.training and torch.is_grad_enabled():
                # 直接传当前 block，不使用捕获循环变量的 lambda。
                # 没有更新 KV cache、运行统计或其他影响结果的副作用。
                x = checkpoint(block, x, allowed, use_reentrant=False,
                               preserve_rng_state=self.preserve_rng_state)
            else:
                x = block(x, allowed)
        return self.head(self.norm(x))


def new_model(cfg=Config(), seed=2501, device=torch.device("cpu"), dtype=torch.float32):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = TinyDecoder(cfg)
    return model.to(device=device, dtype=dtype)

### 3.1 激活检查点：保存边界，反向时重建内部中间值

普通训练会为反向传播保留不少中间量。使用检查点后，不再保留区域内部所有所需中间值，而是在反向传播需要时再次执行区域，重建这些值。

本课的核心调用是：

```python
x = checkpoint(
    block, x, allowed,
    use_reentrant=False,
    preserve_rng_state=True,
)
```

这里直接传入**本轮的 block 对象**，不要写一个不正确捕获循环变量的 `lambda`。反向重算发生得更晚，如果闭包最终指向最后一层，可能出现梯度错误。

显式选用官方推荐的非重入版本 `use_reentrant=False`；即使 embedding 冻结、输入不需要求导，内部可训练参数仍能得到梯度。默认检查不证明数值完全正确，它主要对比重算张量的形状、类型和设备，仍需自行比较数值与梯度。[官方接口与限制](https://docs.pytorch.org/docs/2.10/checkpoint.html)

检查点不是 `detach()` 或 `no_grad()`：后两者可能切断需要的梯度，不能当作节省激活的等价替代。

## 4. 一个窗口的正确更新顺序

```text
窗口开始：zero_grad 一次
    小批次 1：loss_sum / window_tokens → backward
    小批次 2：loss_sum / window_tokens → backward
    ……
窗口结束：unscale 一次 → clip 一次 → step 一次 → scaler.update 一次
```

每次反传后释放当前小批次的图。不要把所有带图的 `loss` 存在列表里、最后再一起 `backward`；那会保留多个图，抵消减少激活的目的。只记录 `loss.detach()` 或 `.item()` 得到的数值。也不需要 `retain_graph=True`。

AMP/FP16 的 scale 必须在同一窗口内不变；不能把已经去缩放的梯度与后续缩放梯度混加。裁剪只处理完成累积并去缩放后的最终梯度。[官方 AMP 梯度累积示例](https://docs.pytorch.org/docs/2.10/notes/amp_examples.html#gradient-accumulation)

优化器步数、学习率调度也应根据你的“更新”定义推进，而不是无意中每个 micro 都推进。本课为了隔离变量不使用学习率调度器，使用固定学习率，并分别统计尝试与实际更新次数。

In [10]:
class CountingAdamW(torch.optim.AdamW):
    def __init__(self, params, **kwargs):
        super().__init__(params, **kwargs)
        self.actual_steps = 0

    def step(self, closure=None):
        result = super().step(closure=closure)
        self.actual_steps += 1
        return result


def update_window(model, optimizer, scaler, window, precision="fp32", max_norm=1.0):
    """一个 effective batch 对应一次尝试更新。loss 分母是整个窗口的有效 token 数。"""
    if not window:
        raise ValueError("窗口不能为空")
    if not math.isfinite(max_norm) or max_norm <= 0:
        raise ValueError("max_norm 必须正且有限")
    total_tokens = sum(valid_tokens(y) for _, y in window)  # 数据尚在 CPU
    if total_tokens == 0:
        raise ValueError("窗口没有有效目标，拒绝除零")
    if any(valid_tokens(y) == 0 for _, y in window):
        raise ValueError("本课不接收完全没有目标的 micro-batch")
    device = next(model.parameters()).device
    model.train()
    optimizer.zero_grad(set_to_none=True)  # 只在窗口开始清零
    before_steps = optimizer.actual_steps
    scale_before = scaler.get_scale()
    numerator = 0.0
    for cpu_x, cpu_y in window:
        x, y = cpu_x.to(device), cpu_y.to(device)
        with autocast_context(precision, device):
            logits = model(x)
            summed = loss_sum(logits, y)
            loss = summed / total_tokens
        if not bool(torch.isfinite(loss.detach())):
            raise FloatingPointError("前向 loss 非有限，停止；GradScaler 不能修复前向溢出")
        scaler.scale(loss).backward()  # 立即反传，释放本 micro-batch 的计算图
        numerator += float(summed.detach())  # 记录数值，不保存 loss 的图
        del x, y, logits, summed, loss
    scaler.unscale_(optimizer)  # 完成全部累积以后，只执行一次
    norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm,
                                   error_if_nonfinite=not scaler.is_enabled())
    scaler.step(optimizer)      # 如有非有限梯度，enabled scaler 可跳过本窗口
    scaler.update()             # scale 在同一个窗口内必须保持不变
    norm_value = float(norm)
    return {"loss": numerator/total_tokens, "valid_tokens": total_tokens,
            "samples": sum(x.shape[0] for x, _ in window), "microbatches": len(window),
            "grad_norm_before_clip": norm_value if math.isfinite(norm_value) else None,
            "scale_before": scale_before, "scale_after": scaler.get_scale(),
            "optimizer_ran": optimizer.actual_steps > before_steps}


@torch.no_grad()
def evaluate(model, rows, batch_size=8):
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    total, n, correct = 0.0, 0, 0
    try:
        for x, y in iter_microbatches(rows, batch_size):
            x, y = x.to(device), y.to(device)
            with autocast_context("fp32", device):
                logits = model(x)
                total += float(loss_sum(logits, y))
            active = y != IGNORE
            n += int(active.sum())
            correct += int(((logits.argmax(-1) == y) & active).sum())
        if n == 0:
            raise ValueError("评估集无有效目标")
        return {"loss": total/n, "accuracy": correct/n, "valid_tokens": n}
    finally:
        model.train(was_training)

### 4.1 把误差定位在优化器之前

不能只看最终 loss 相似就断言实现正确。我们在 **CPU、FP64、Dropout=0** 条件下，先比较所有参数的梯度，再比较一次相同的 SGD 更新。

对照包含：整批 7 条；拆成 3、3、1；拆批同时启用检查点；故意平均各 micro 的均值。所有输入顺序和初始参数相同。前 3 条是不同实现，最后一条是故意写错的训练目标。

In [11]:
def gradient_vector(model):
    if any(p.grad is None for p in model.parameters() if p.requires_grad):
        raise AssertionError("存在可训练参数没有梯度")
    return torch.cat([p.grad.detach().reshape(-1) for p in model.parameters() if p.requires_grad])


def backward_window(model, batches, wrong_mean=False):
    model.zero_grad(set_to_none=True)
    total = sum(valid_tokens(y) for _, y in batches)
    if total <= 0:
        raise ValueError("无有效目标")
    for x, y in batches:
        summed = loss_sum(model(x), y)
        # wrong_mean 分支只用于故障反例，不用于实际训练。
        denom = valid_tokens(y)*len(batches) if wrong_mean else total
        (summed / denom).backward()
    return gradient_vector(model).clone()


def equivalence_demo():
    cfg = Config(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32)
    train, _ = make_data()
    rows = train[:7]   # micro 3,3,1；每条有效长度也不同
    base = new_model(cfg, dtype=torch.float64)
    initial = copy.deepcopy(base.state_dict())
    variants = {}
    for name, micro, ac, wrong in [("full", 7, False, False), ("accum", 3, False, False),
                                    ("checkpoint", 3, True, False), ("wrong_mean", 3, False, True)]:
        m = new_model(cfg, dtype=torch.float64)
        m.load_state_dict(initial)
        m.activation_checkpointing = ac
        batches = list(iter_microbatches(rows, micro))
        grad = backward_window(m, batches, wrong)
        norm = nn.utils.clip_grad_norm_(m.parameters(), 0.5)
        # 用 SGD 做一步干净对照；多次 AdamW 训练另外独立报告。
        optimizer = torch.optim.SGD(m.parameters(), lr=0.1)
        optimizer.step()
        params = torch.cat([p.detach().reshape(-1) for p in m.parameters()])
        variants[name] = {"gradient": grad, "parameters": params, "norm": float(norm)}
    ref = variants["full"]
    return {"micro_token_counts": [valid_tokens(y) for _, y in iter_microbatches(rows, 3)],
            "dtype": "float64", "dropout": 0.0,
            "comparisons": {name: {
                "gradient_max_abs": float((item["gradient"]-ref["gradient"]).abs().max()),
                "parameter_max_abs_after_one_step": float((item["parameters"]-ref["parameters"]).abs().max())}
                for name, item in variants.items() if name != "full"}}


def step_scope_demo():
    """两次 backward 一次 step，不等于每个 micro-batch 各 step 一次。"""
    p = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
    opt = torch.optim.SGD([p], lr=0.1)
    for target in (1.0, 3.0):
        ((p-target).square()/2).backward()
    opt.step()
    correct = float(p.detach())
    q = nn.Parameter(torch.tensor(0.0, dtype=torch.float64))
    opt2 = torch.optim.SGD([q], lr=0.1)
    for target in (1.0, 3.0):
        opt2.zero_grad()
        ((q-target).square()/2).backward()
        opt2.step()
    return {"once_per_window": correct, "wrong_step_each_micro": float(q.detach())}

In [12]:
EQUIVALENCE = equivalence_demo()
print("各 micro 的有效 token 数:", EQUIVALENCE["micro_token_counts"])
for name, errors in EQUIVALENCE["comparisons"].items():
    print(name, errors)
print("每个 micro 都 step 的反例:", step_scope_demo())

各 micro 的有效 token 数: [25, 19, 5]
accum {'gradient_max_abs': 8.326672684688674e-17, 'parameter_max_abs_after_one_step': 2.7755575615628914e-17}
checkpoint {'gradient_max_abs': 8.326672684688674e-17, 'parameter_max_abs_after_one_step': 2.7755575615628914e-17}
wrong_mean {'gradient_max_abs': 0.2089032620285551, 'parameter_max_abs_after_one_step': 0.009521616558635398}
每个 micro 都 step 的反例: {'once_per_window': 0.4, 'wrong_step_each_micro': 0.39}


**读数方法：**正常累积和检查点的梯度应在浮点容差内与整批相同；故意使用错误分母的梯度应有明显偏差。这里每个 micro 的有效 token 数是 **25、19、5**，恰好能揭露“均值再平均”的问题。

另一个标量反例中，窗口结束再更新得到 `0.4`，每个 micro 都更新得到 `0.39`。即使总学习率看起来相近，第二个 micro 已经在不同参数上计算，因而不再等价。

## 5. Dropout：前向一样，梯度也可能错

重算如果重新抽取另一张 Dropout mask，会对错误的前向函数求梯度。`preserve_rng_state=True` 会暂存并恢复相关随机数状态，使这次重算与原前向保持一致。[PyTorch RNG 说明](https://docs.pytorch.org/docs/2.10/checkpoint.html)

下面专门开启 `dropout=0.4`，使用相同 micro-batch 和随机种子，比较：普通计算、保存 RNG 的检查点、不保存 RNG 的检查点。

这项比较与“整批和拆批用了不同的随机 mask”不是同一件事：这里输入、调用粒度都相同，唯一变化是重计算时是否保存 RNG。

In [13]:
def checkpoint_rng_demo():
    cfg = Config(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32, dropout=0.4)
    x, y = collate(make_data()[0][:4])
    state = copy.deepcopy(new_model(cfg).state_dict())
    runs = {}
    for name, ac, preserve in [("plain", False, True), ("preserved", True, True),
                                ("not_preserved", True, False)]:
        model = new_model(cfg)
        model.load_state_dict(state)
        model.activation_checkpointing, model.preserve_rng_state = ac, preserve
        entries = [0 for _ in model.blocks]
        handles = []
        for index, block in enumerate(model.blocks):
            # 默认参数绑定 index；hook 只计数，不参与模型计算。
            def count(_module, _args, index=index):
                entries[index] += 1
            handles.append(block.register_forward_pre_hook(count))
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(2511)
            logits = model(x)
            (loss_sum(logits, y)/valid_tokens(y)).backward()
            rng_after = torch.get_rng_state().clone()
        for handle in handles:
            handle.remove()
        runs[name] = {"output": logits.detach(), "gradient": gradient_vector(model),
                      "rng": rng_after, "block_entries": entries}
    ref = runs["plain"]
    return {"comparisons": {name: {
                "forward_max_abs": float((r["output"]-ref["output"]).abs().max()),
                "gradient_max_abs": float((r["gradient"]-ref["gradient"]).abs().max()),
                "rng_matches_plain": bool(torch.equal(r["rng"], ref["rng"])),
                "block_entries": r["block_entries"]}
             for name, r in runs.items() if name != "plain"},
            "plain_block_entries": ref["block_entries"],
            "note": "非重入可提前停止重计算；调用入口计数不等于完整 FLOPs。"}


def scaler_window_demo():
    """真实 CPU GradScaler 控制实验：只在窗口结束去缩放/更新；第二个窗口故意破坏梯度。"""
    p = nn.Parameter(torch.tensor(1.0))
    opt = torch.optim.SGD([p], lr=0.1)
    scaler = torch.amp.GradScaler("cpu", init_scale=128.0, growth_interval=200)
    called = {"n": 0}
    def after_step(*_):
        called["n"] += 1
    handle = opt.register_step_post_hook(after_step)
    result = []
    for inject in (False, True):
        opt.zero_grad(set_to_none=True)
        before, prev = float(p.detach()), called["n"]
        scales = []
        for _ in range(3):
            scales.append(scaler.get_scale())
            scaler.scale(p.square()/3).backward()
        if inject:
            p.grad.fill_(float("inf"))
        scaler.unscale_(opt)
        if bool(torch.isfinite(p.grad).all()):
            nn.utils.clip_grad_norm_([p], 10.0)
        scaler.step(opt)
        scaler.update()
        result.append({"scales_within_window": scales, "scale_after": scaler.get_scale(),
                       "optimizer_ran": called["n"] > prev, "before": before,
                       "after": float(p.detach()), "injected_inf": inject})
    handle.remove()
    return result

In [14]:
RNG_RESULT = checkpoint_rng_demo()
print(json.dumps(RNG_RESULT, ensure_ascii=False, indent=2))

{
  "comparisons": {
    "preserved": {
      "forward_max_abs": 0.0,
      "gradient_max_abs": 0.0,
      "rng_matches_plain": true,
      "block_entries": [
        2,
        2
      ]
    },
    "not_preserved": {
      "forward_max_abs": 0.0,
      "gradient_max_abs": 0.061077967286109924,
      "rng_matches_plain": false,
      "block_entries": [
        2,
        2
      ]
    }
  },
  "plain_block_entries": [
    1,
    1
  ],
  "note": "非重入可提前停止重计算；调用入口计数不等于完整 FLOPs。"
}


预期能看到：不保存 RNG 时，**原始前向输出仍可能完全相同，但梯度已经不同**。因此不能只做前向 `allclose`。

普通块的入口通常调用一次；检查点使其反向时再次进入。本实验只记录入口次数，**不等于重做了恰好两倍 FLOPs**，因为非重入实现可在所需中间量重建完成后提前停止。

检查点区域应避免改变会影响结果的全局变量、追加缓存、运行统计和移动到新设备。重复执行有副作用的代码会使原前向与重算不同。本课没有 BatchNorm、KV cache 或跨设备计算。

### 5.1 AMP 控制实验：坏梯度应跳过整个窗口

这一小实验在 CPU 使用真实 GradScaler，验证状态控制，不把它当作 CUDA FP16 性能测试。一个窗口含 3 次 backward；第二个窗口在去缩放之前故意注入 `inf` 梯度。

要求：窗口内 scale 不变；只在窗口末调用 unscale/step/update；非有限梯度时参数不动；随后 scale 降低。只检查前向 loss 是否有限是不够的。

In [15]:
for i, row in enumerate(scaler_window_demo(), start=1):
    print(f"窗口 {i}:", row)

窗口 1: {'scales_within_window': [128.0, 128.0, 128.0], 'scale_after': 128.0, 'optimizer_ran': True, 'before': 1.0, 'after': 0.800000011920929, 'injected_inf': False}
窗口 2: {'scales_within_window': [128.0, 128.0, 128.0], 'scale_after': 64.0, 'optimizer_ran': False, 'before': 0.800000011920929, 'after': 0.800000011920929, 'injected_inf': True}


## 6. 内存观察：小 micro 与激活检查点分别减少什么？

为了在没有 GPU 时也观察机制，本课使用 `saved_tensors_hooks` 统计**前向为反向保留的底层存储**。排除参数、buffer 和输入，重复视图只按同一存储算一次；`pack` 返回 `detach` 视图，不复制张量也不保留额外图。[Autograd saved-tensor hooks](https://docs.pytorch.org/docs/stable/notes/autograd.html)

这个统计包含内部 mask、logits 等，属于教学代理：

- 不是进程内存；不是 GPU allocator 峰值；不能代替 `nvidia-smi`。
- 累积路径是多个小图先后执行，报告取各图的**最大值**，不能把它们相加充当同时峰值。
- 真正训练还需要权重、梯度、优化器、边界激活、临时工作区等；本课后面提供可选 CUDA 实测。

下方对照固定有效样本为 12 条，三层、隐藏维度 48。这里只比较“整批 12”、“小批 3”和“小批 3 + 检查点”。

In [16]:
def storage_key(tensor):
    storage = tensor.untyped_storage()
    return (str(tensor.device), storage.data_ptr(), storage.nbytes())


def saved_storage_snapshot(model, x, y):
    """记录前向 hooks 看到的去重存储载荷；排除模型参数/buffer/输入底层存储。

    返回 pack 的是 detach view，不复制、不保留额外图；重复视图只计一次。
    不在 hook 范围中执行 backward，避免将重计算新保存内容再次累加。
    这个口径包含内部 mask/logits 等，不是完整训练峰值；不用于性能计时。
    """
    excluded = {storage_key(t) for t in [*model.parameters(), *model.buffers(), x, y]}
    storages, calls = set(), {"n": 0}
    def pack(t):
        calls["n"] += 1
        key = storage_key(t)
        if t.numel() and key not in excluded:
            storages.add(key)
        return t.detach()
    def unpack(t):
        return t
    model.zero_grad(set_to_none=True)
    with torch.autograd.graph.saved_tensors_hooks(pack, unpack):
        loss = loss_sum(model(x), y)/valid_tokens(y)
    snapshot = {"saved_tensor_hook_calls": calls["n"], "unique_nonpersistent_storages": len(storages),
                "unique_saved_nonpersistent_bytes": sum(k[2] for k in storages)}
    loss.backward()
    return snapshot


def memory_proxy_demo():
    cfg = Config(d_model=48, ff_dim=128, n_layers=3)
    state = copy.deepcopy(new_model(cfg).state_dict())
    rows = make_data()[0][:12]
    result = {}
    for name, micro, ac in [("full", 12, False), ("accum", 3, False), ("checkpoint", 3, True)]:
        model = new_model(cfg)
        model.load_state_dict(state)
        model.activation_checkpointing = ac
        snapshots = [saved_storage_snapshot(model, x, y) for x, y in iter_microbatches(rows, micro)]
        result[name] = {"micro_size": micro, "activation_checkpointing": ac,
                        "max_single_forward_saved_bytes": max(r["unique_saved_nonpersistent_bytes"] for r in snapshots),
                        "per_micro_forward": snapshots}
    result["note"] = "CPU 前向 saved-tensors 去重存储代理；排除参数/输入；不是峰值显存，不相加充当累积窗口峰值。"
    return result

In [17]:
MEMORY_PROXY = memory_proxy_demo()
for name in ("full", "accum", "checkpoint"):
    row = MEMORY_PROXY[name]
    n = row["max_single_forward_saved_bytes"]
    print(f"{name:12s}: {n:,} bytes，约 {n/1024:.2f} KiB")
print(MEMORY_PROXY["note"])

full        : 1,377,220 bytes，约 1344.94 KiB
accum       : 346,252 bytes，约 338.14 KiB
checkpoint  : 33,187 bytes，约 32.41 KiB
CPU 前向 saved-tensors 去重存储代理；排除参数/输入；不是峰值显存，不相加充当累积窗口峰值。


### 6.1 别把权重放不下的问题当成激活问题

本课全参数训练仍然保留 FP32 参数、FP32 梯度和 Adam 状态。检查点主要改变激活的保存策略，不会把这些常驻状态都压缩掉。

因此，“3060 上能做梯度累积”不意味着任意大模型都能全参数训练。如果模型权重和优化器本身就超出预算，还要研究参数高效微调、低比特加载、状态分片或卸载。也不要用每步 `torch.cuda.empty_cache()` 来代替分析正在被引用的张量。

## 7. 完整三路训练：只改变计算的组织方式

默认三路共享初始参数和每轮样本顺序，窗口对应样本相同，学习率、裁剪、AdamW 设置也相同：

| 路径 | 每次样本数 | 累积上限 | 检查点 |
|---|---:|---:|---|
| `full` | 12 | 1 | 否 |
| `accum` | 3 | 4 | 否 |
| `checkpoint` | 3 | 4 | 是 |

43 条训练、17 条验证，8 轮，每轮 4 个窗口，总共 **32 次尝试更新**。尾窗口包含 7 条样本，不丢弃。

训练关闭 Dropout，使用固定最后一轮而不是按验证集挑选“最好”的结果。验证统一采用 FP32。各路径最终数值应接近，但多次 AdamW 更新会放大不同舍入，仍不要求逐位一致。

In [18]:
def capability_probe(precision, device, cfg=Config()):
    if precision == "amp_fp16" and device.type != "cuda":
        raise ValueError("本课完整 AMP/FP16 训练只在 CUDA 路径运行；CPU 请用 fp32/amp_bf16")
    if precision == "amp_bf16" and device.type == "cuda" and not torch.cuda.is_bf16_supported():
        raise RuntimeError("CUDA 环境未报告 BF16 支持")
    model = new_model(cfg, device=device)
    model.activation_checkpointing = True
    x, y = collate(make_data()[0][:3])
    opt = CountingAdamW(model.parameters(), lr=0.003, foreach=False)
    record = update_window(model, opt, make_scaler(precision, device), [(x, y)], precision)
    if not record["optimizer_ran"]:
        raise RuntimeError("完整 checkpoint + AMP 探测跳过了更新，请检查精度/scale")
    return {"precision": precision, "device": str(device), "full_forward_backward_probe_passed": True}


def train_variant(cfg, state, rows, validation, epochs, strategy, precision, device):
    if strategy not in ("full", "accum", "checkpoint"):
        raise ValueError("未知 strategy")
    model = new_model(cfg, device=device)
    model.load_state_dict(state)
    model.activation_checkpointing = strategy == "checkpoint"
    micro, accumulation = (12, 1) if strategy == "full" else (3, 4)
    opt = CountingAdamW(model.parameters(), lr=0.003, weight_decay=0.01, foreach=False)
    scaler = make_scaler(precision, device)
    initial = evaluate(model, validation)
    history, updates = [], []
    for epoch in range(epochs):
        order = torch.randperm(len(rows), generator=torch.Generator().manual_seed(2520+epoch)).tolist()
        for win in accumulation_windows(iter_microbatches(rows, micro, order), accumulation):
            row = update_window(model, opt, scaler, win, precision)
            row["epoch"] = epoch+1
            updates.append(row)
        history.append({"epoch": epoch+1, "train": evaluate(model, rows),
                        "validation": evaluate(model, validation), "actual_updates": opt.actual_steps})
    result = {"strategy": strategy, "precision": precision, "micro_size": micro,
              "accumulation_steps": accumulation, "dropout": cfg.dropout,
              "initial_validation": initial, "final_validation": history[-1]["validation"],
              "attempted_updates": len(updates), "actual_updates": opt.actual_steps,
              "effective_samples_seen": sum(r["samples"] for r in updates),
              "initial_state_sha256": fingerprint(state), "history": history, "updates": updates,
              "final_state_sha256": fingerprint(model.state_dict()),
              "note": "固定最后一轮，不选择最佳验证轮次；合成数据、单种子、无独立测试集。"}
    saved_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return result, saved_state

In [19]:
def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def resource_benchmark(device, precision="fp32", length=64, repeats=3):
    positive_int(length, "length")
    positive_int(repeats, "repeats")
    cfg = Config(d_model=64, n_heads=4, n_kv_heads=2, ff_dim=160,
                 n_layers=3, max_len=max(9, length))
    g = torch.Generator().manual_seed(2530)
    x = torch.randint(4, cfg.vocab_size, (8, length), generator=g)
    x[:, 0] = BOS
    y = torch.randint(4, cfg.vocab_size, (8, length), generator=g)
    state = copy.deepcopy(new_model(cfg).state_dict())
    plans = {"full": (8, False), "accum": (2, False), "checkpoint": (2, True)}
    records = {key: [] for key in plans}
    # 每个 trial 重建独立模型/优化器，同初始状态，同预热输入。
    # 顺序轮换，计时前同步；输入 CPU->device 与有效目标统计在测量外。
    for trial in range(repeats):
        names = list(plans)
        names = names[trial % len(names):] + names[:trial % len(names)]
        for name in names:
            model = opt = scaler = window = a = b = None
            try:
                micro, ac = plans[name]
                model = new_model(cfg, device=device)
                model.load_state_dict(state)
                model.activation_checkpointing = ac
                opt = CountingAdamW(model.parameters(), lr=0.003, foreach=False)
                scaler = make_scaler(precision, device)
                # 预热两次，也让 Adam 状态完成分配。
                cpu_window = [(x[i:i+micro], y[i:i+micro]) for i in range(0, 8, micro)]
                for _ in range(2):
                    update_window(model, opt, scaler, cpu_window, precision)
                window = [(a.to(device), b.to(device)) for a, b in cpu_window]
                n = 8*length
                opt.zero_grad(set_to_none=True)
                synchronize(device)
                baseline = torch.cuda.memory_allocated(device) if device.type == "cuda" else None
                if device.type == "cuda":
                    torch.cuda.reset_peak_memory_stats(device)
                start = time.perf_counter()
                # 无逐 micro 的 .item()/CPU 同步，计时是整个更新；不保存各 micro 的图。
                for a, b in window:
                    with autocast_context(precision, device):
                        part = loss_sum(model(a), b)/n
                    scaler.scale(part).backward()
                    del part
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0,
                                         error_if_nonfinite=not scaler.is_enabled())
                before = opt.actual_steps
                scaler.step(opt)
                scaler.update()
                synchronize(device)
                elapsed = (time.perf_counter()-start)*1000
                peak = torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None
                records[name].append({"trial": trial, "status": "ok", "ms_per_effective_update": elapsed,
                                      "baseline_allocated_bytes": baseline, "peak_allocated_bytes": peak,
                                      "incremental_peak_bytes": peak-baseline if peak is not None else None,
                                      "peak_reserved_bytes": torch.cuda.max_memory_reserved(device) if device.type == "cuda" else None,
                                      "optimizer_ran": opt.actual_steps > before})
            except torch.OutOfMemoryError as exc:
                records[name].append({"trial": trial, "status": "oom", "reason": str(exc)[:500]})
            finally:
                model = opt = scaler = window = a = b = None
                gc.collect()
                if device.type == "cuda":
                    torch.cuda.empty_cache()
    summary = {}
    for name, rows in records.items():
        good = [r for r in rows if r["status"] == "ok" and r["optimizer_ran"]]
        summary[name] = {"successful_trials": len(good),
                         "median_ms_per_effective_update": statistics.median(r["ms_per_effective_update"] for r in good) if good else None,
                         "median_peak_allocated_bytes": statistics.median(r["peak_allocated_bytes"] for r in good) if good and device.type == "cuda" else None}
    return {"device": str(device), "precision": precision, "config": asdict(cfg),
            "effective_batch": 8, "length": length, "repeats": repeats,
            "rows": records, "summary": summary,
            "scope": "固定输入的一次完整优化更新：forward/backward/clip/step；预热后测；无 H2D；非推理延迟。",
            "limitations": "微模型短测试，非硬件排名；CPU 显存字段 null；allocated 峰值不等于 nvidia-smi；未调用 compile。"}

In [20]:
class Checks:
    def __init__(self):
        self.names = []

    def check(self, condition, name):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def close(self, actual, expected, name, atol=1e-8, rtol=1e-6):
        a, b = torch.as_tensor(actual), torch.as_tensor(expected)
        torch.testing.assert_close(a, b.to(a), atol=atol, rtol=rtol, msg=name)
        self.names.append(name)

    def raises(self, fn, name, kind=ValueError):
        try:
            fn()
        except kind:
            self.names.append(name)
        else:
            raise AssertionError(f"应该拒绝：{name}")


def run_checks():
    c = Checks()
    train, val = make_data()
    c.check(len(train) == 43 and len(val) == 17, "固定数据划分 43/17")
    c.check(not set(map(tuple, train)) & set(map(tuple, val)), "训练验证不重合")
    c.check(len(set(len(x) for x in train)) > 1, "训练样本确实变长")
    c.check(all(row[0] == BOS and row[-1] == EOS for row in train+val), "完整样本 BOS/EOS")
    x, y = collate(train[:7])
    c.check(x.shape == y.shape, "输入标签同形")
    c.check(x.dtype == torch.long and y.dtype == torch.long, "token 为 long")
    c.check(torch.equal(x[0, 1:len(train[0])-1], y[0, :len(train[0])-2]), "标签只移位一次")
    c.check(valid_tokens(y) == sum(len(row)-1 for row in train[:7]), "有效目标计数包含 EOS")
    c.check(bool((y == IGNORE).any()), "标签存在 padding 忽略值")
    c.raises(lambda: collate([]), "拒绝空数据")
    for n in (0, -1, True):
        c.raises(lambda n=n: list(accumulation_windows([], n)), f"拒绝非法累积步数 {n}")
    c.check(list(accumulation_windows([], 4)) == [], "空迭代器不制造窗口")
    windows = window_summary(train)
    c.check([r["microbatches"] for r in windows] == [4, 4, 4, 3], "保留不足 K 的尾窗口")
    c.check([r["samples"] for r in windows] == [12, 12, 12, 7], "最后窗口不是假装 12 条")
    c.check(sum(r["tokens"] for r in windows) == sum(len(r)-1 for r in train), "整轮无丢样本目标")
    d = denominator_demo()
    c.close(d["correct_token_mean"], 1.8, "正确 token 加权均值")
    c.close(d["wrong_mean_of_means"], 5.0, "反例暴露错误平均")
    e = equivalence_demo()
    for name in ("accum", "checkpoint"):
        c.check(e["comparisons"][name]["gradient_max_abs"] < 2e-10, f"{name} 所有参数梯度等价")
        c.check(e["comparisons"][name]["parameter_max_abs_after_one_step"] < 2e-10, f"{name} 一步裁剪更新等价")
    c.check(e["comparisons"]["wrong_mean"]["gradient_max_abs"] > 1e-5, "错误分母确实改变梯度")
    scope = step_scope_demo()
    c.close(scope["once_per_window"], 0.4, "窗口一次更新的标量答案")
    c.close(scope["wrong_step_each_micro"], 0.39, "每小批更新不是累积")
    rng = checkpoint_rng_demo()
    for key in ("forward_max_abs", "gradient_max_abs"):
        c.check(rng["comparisons"]["preserved"][key] < 1e-7, f"Dropout 保存 RNG：{key}")
    c.check(rng["comparisons"]["preserved"]["rng_matches_plain"], "重算不额外消耗外部 RNG")
    c.check(rng["comparisons"]["not_preserved"]["forward_max_abs"] == 0, "不保存 RNG 时前向仍可能一致")
    c.check(rng["comparisons"]["not_preserved"]["gradient_max_abs"] > 1e-5, "不保存 RNG 的梯度错误被发现")
    c.check(all(n == 1 for n in rng["plain_block_entries"]), "普通块执行一次")
    c.check(all(n == 2 for n in rng["comparisons"]["preserved"]["block_entries"]), "检查点确实触发重算")
    # 验证实际训练函数，不只验证数学参考函数，防止二者实现脱节。
    update_cfg = Config(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32)
    update_full = new_model(update_cfg, dtype=torch.float64)
    update_micro = copy.deepcopy(update_full)
    opt_full = CountingAdamW(update_full.parameters(), lr=0.003, foreach=False)
    opt_micro = CountingAdamW(update_micro.parameters(), lr=0.003, foreach=False)
    for p in update_micro.parameters():
        p.grad = torch.full_like(p, 5.0)  # 确认窗口开始会清理旧梯度
    report_full = update_window(update_full, opt_full, make_scaler("fp32", torch.device("cpu")), [collate(train[:7])])
    report_micro = update_window(update_micro, opt_micro, make_scaler("fp32", torch.device("cpu")), list(iter_microbatches(train[:7], 3)))
    c.close(gradient_vector(update_full), gradient_vector(update_micro), "实际 update_window 的梯度等价且清除旧梯度", atol=1e-10)
    c.check(max(float((p-q).detach().abs().max()) for p, q in zip(update_full.parameters(), update_micro.parameters())) < 1e-9,
            "实际 AdamW 一步参数等价")
    c.close(report_full["loss"], report_micro["loss"], "实际报告按有效 token 汇总", atol=1e-10)
    c.check(opt_micro.actual_steps == opt_full.actual_steps == 1, "实际窗口更新只调用一次 step")
    scales = scaler_window_demo()
    c.check(scales[0]["scales_within_window"] == [128.0]*3, "窗口内 scale 保持不变")
    c.check(scales[0]["optimizer_ran"] and not scales[1]["optimizer_ran"], "非有限梯度跳过整个窗口")
    c.close(scales[1]["before"], scales[1]["after"], "跳过时参数不变")
    c.check(scales[1]["scale_after"] == 64.0, "窗口结束才降 scale")
    cfg = Config(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32)
    model = new_model(cfg, dtype=torch.float64)
    c.raises(lambda: model(x.float()), "拒绝浮点 token")
    c.raises(lambda: model(torch.tensor([[0, 1]])), "拒绝左补齐")
    c.raises(lambda: model(torch.tensor([[1, 0, 2]])), "拒绝中间 padding")
    c.raises(lambda: model(torch.ones(1, 33, dtype=torch.long)), "拒绝长度越界")
    c.check(sum(p.numel() for p in new_model().parameters()) == 52464, "参数总数 52464")
    state_hash = fingerprint(model.state_dict())
    model.eval()
    with torch.no_grad():
        out = model(x)
        short = model(x[:1, :len(train[0])-1])
    c.close(out[:1, :short.shape[1]], short, "右侧 padding 不影响有效输出", atol=1e-10)
    # 改用固定无 padding 序列，单独验证因果性。
    fixed = torch.tensor([[BOS, 4, 5, 4, 5]])
    changed = fixed.clone(); changed[0, -1] = 7
    with torch.no_grad():
        c.close(model(fixed)[:, :-1], model(changed)[:, :-1], "因果性：未来不影响前缀", atol=1e-10)
    plain = copy.deepcopy(model)
    checked = copy.deepcopy(model)
    checked.activation_checkpointing = True
    plain.train(); checked.train()
    g1 = backward_window(plain, [(x, y)])
    g2 = backward_window(checked, [(x, y)])
    c.close(g1, g2, "整个模型 checkpoint 梯度同形同值", atol=1e-10)
    c.check(fingerprint(model.state_dict()) == state_hash, "梯度实验不修改原参数")
    # 冻结 embedding：非重入版本仍能训练内部 block 参数，不依赖输入 requires_grad。
    frozen = copy.deepcopy(model); frozen.train(); frozen.activation_checkpointing = True
    frozen.embedding.weight.requires_grad_(False)
    loss_sum(frozen(x), y).backward()
    c.check(frozen.embedding.weight.grad is None, "冻结 embedding 没有梯度")
    c.check(all(p.grad is not None for n, p in frozen.named_parameters() if n != "embedding.weight"), "冻结输入不切断内部参数梯度")
    eval_model = copy.deepcopy(model); eval_model.activation_checkpointing = True
    entries = []
    handle = eval_model.blocks[0].register_forward_pre_hook(lambda *_: entries.append(1))
    with torch.no_grad():
        eval_model(x)
    handle.remove()
    c.check(len(entries) == 1, "eval no_grad 不启用检查点重算")
    small = new_model(cfg)
    opt = CountingAdamW(small.parameters(), lr=0.003, foreach=False)
    scaler = make_scaler("fp32", torch.device("cpu"))
    for win in accumulation_windows(iter_microbatches(train, 3), 4):
        update_window(small, opt, scaler, win)
    c.check(opt.actual_steps == 4, "15 个 micro-batch 只更新 4 次")
    c.check(all(int(s["step"]) == 4 for s in opt.state.values()), "Adam 计数按窗口不是 micro")
    c.raises(lambda: update_window(small, opt, scaler, []), "拒绝空窗口")
    c.raises(lambda: update_window(small, opt, scaler, [(x, torch.full_like(y, IGNORE))]), "拒绝零有效目标窗口")
    small.train()
    evaluate(small, val)
    c.check(small.training, "评估恢复训练模式")
    v1, v2 = evaluate(small, val, 1), evaluate(small, val, 8)
    c.close(v1["loss"], v2["loss"], "验证按 token 汇总不依赖 batch size", atol=2e-6)
    proxy = memory_proxy_demo()
    c.check(proxy["checkpoint"]["max_single_forward_saved_bytes"] < proxy["accum"]["max_single_forward_saved_bytes"], "检查点减少 hooks 保留载荷")
    c.check(proxy["accum"]["max_single_forward_saved_bytes"] < proxy["full"]["max_single_forward_saved_bytes"], "小 micro 减少单图保留载荷")
    c.check(set(model.state_dict()) == set(checked.state_dict()), "检查点开关不增加持久参数或 KV cache")
    c.check(all(p.dtype == torch.float64 for p in model.parameters()), "参考参数 dtype 未改变")
    print(f"通过固定检查：{len(c.names)} 项（CPU 机制参考）。")
    return {"passed": len(c.names), "names": c.names, "device": "cpu",
            "equivalence": e, "dropout_checkpoint": rng, "scaler_windows": scales}

In [21]:
def run_experiment(output_dir="lesson_25_outputs", device_name="cpu", precision="fp32",
                   epochs=8, strategies=("full", "accum", "checkpoint"), benchmark=False,
                   bench_length=64, checks=True):
    positive_int(epochs, "epochs")
    device = choose_device(device_name)
    if not strategies or any(s not in ("full", "accum", "checkpoint") for s in strategies):
        raise ValueError("strategies 需包含 full/accum/checkpoint 中的一项或多项")
    cfg = Config()
    output = Path(output_dir).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=True)
    with reference_precision():
        tested = run_checks() if checks else {"passed": None, "note": "未运行固定检查"}
        probe = capability_probe(precision, device, cfg)
        train, val = make_data()
        state = copy.deepcopy(new_model(cfg).state_dict())
        trained, states = {}, {}
        for strategy in dict.fromkeys(strategies):
            print(f"训练路径：{strategy} / {precision} / {device}")
            result, saved = train_variant(cfg, state, train, val, epochs, strategy, precision, device)
            payload = {"lesson": 25, "config": asdict(cfg), "strategy": strategy, "precision": precision,
                       "state_dict": saved, "note": "仅恢复模型；不含完整优化器/RNG/数据进度，不能精确断点续训"}
            path = output / f"{strategy}.pt"
            torch.save(payload, path)
            loaded = torch.load(path, map_location="cpu", weights_only=True)
            rebuilt = new_model(Config(**loaded["config"]))
            rebuilt.load_state_dict(loaded["state_dict"], strict=True)
            result["reload_state_equal"] = fingerprint(rebuilt.state_dict()) == fingerprint(saved)
            if not result["reload_state_equal"]:
                raise AssertionError("保存重载权重不一致")
            trained[strategy], states[strategy] = result, saved
        comparisons = {}
        if "full" in states:
            for name, saved in states.items():
                if name != "full":
                    comparisons[name] = {"final_parameter_max_abs_vs_full": max(
                        float((saved[k]-states["full"][k]).abs().max()) for k in saved if saved[k].is_floating_point()),
                        "interpretation": "FP32/AMP 舍入可经多次 AdamW 更新放大；不能要求逐位相同。FP64 一步机制实验另查。"}
        report = {"lesson": 25, "environment": {"python": platform.python_version(),
                    "torch": str(torch.__version__), "device": str(device), "precision": precision,
                    "cuda_available": torch.cuda.is_available(), "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
                    "threads": torch.get_num_threads()},
                  "config": asdict(cfg), "parameter_count": sum(p.numel() for p in new_model(cfg).parameters()),
                  "checks": tested, "capability_probe": probe, "denominator_demo": denominator_demo(),
                  "data": {"train": train, "validation": val, "windows": window_summary(train)},
                  "equivalence": equivalence_demo(), "step_scope": step_scope_demo(),
                  "checkpoint_rng": checkpoint_rng_demo(), "saved_storage_proxy": memory_proxy_demo(),
                  "training": trained, "comparisons": comparisons,
                  "resource_benchmark": resource_benchmark(device, precision, bench_length) if benchmark else None,
                  "sources": SOURCES,
                  "limitations": ["单设备，无 DDP/ZeRO/CPU offload；不使用 KV cache；没有精确恢复训练。",
                      "默认 Dropout=0，无 BatchNorm/跨样本损失；严格核对另用 CPU FP64。",
                      "CPU hooks 载荷不是 GPU 峰值；CUDA 未运行时显存数据为空，不填 0。",
                      "策略比较固定有效窗口、初始化和数据顺序；合成 token 实验不是语言模型能力评测。"]}
    def write(name, value):
        (output / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    write("experiment.json", report)
    write("training.json", trained)
    write("memory_report.json", {"saved_storage_proxy": report["saved_storage_proxy"],
                                 "resource_benchmark": report["resource_benchmark"]})
    print("\n最终验证指标（统一 FP32 评估）：")
    for key, row in trained.items():
        print(f"{key:12s} loss={row['final_validation']['loss']:.6f}，更新 {row['actual_updates']}/{row['attempted_updates']}")
    print(f"输出目录：{output}")
    return report

### 7.1 运行完整实验

下面会运行固定检查、设备/精度探测和三路训练，生成目录中同名报告与模型文件。**重新运行会覆盖本课自己的同名生成文件**，不读取或修改其他课程文件。

默认只做正确性与训练。运行资源基准时先把顶部 `RUN_RESOURCE_BENCHMARK` 改为 `True`。显式 `DEVICE="cuda"` 不可用会报错；不会静默把 CPU 结果伪装成 GPU 结果。

In [22]:
REPORT = run_experiment(
    output_dir=OUTPUT_DIR,
    device_name=DEVICE,
    precision=PRECISION,
    epochs=EPOCHS,
    strategies=STRATEGIES,
    benchmark=RUN_RESOURCE_BENCHMARK,
    bench_length=BENCH_LENGTH,
    checks=True,
)

通过固定检查：63 项（CPU 机制参考）。
训练路径：full / fp32 / cpu


训练路径：accum / fp32 / cpu


训练路径：checkpoint / fp32 / cpu



最终验证指标（统一 FP32 评估）：
full         loss=2.572053，更新 32/32
accum        loss=2.572052，更新 32/32
checkpoint   loss=2.572052，更新 32/32
输出目录：/mnt/data/lesson_25_validation/notebook_run/lesson_25_outputs


In [23]:
print("参数量:", REPORT["parameter_count"])
for name, row in REPORT["training"].items():
    print(
        f"{name:12s} | 初始验证 {row['initial_validation']['loss']:.6f} "
        f"→ 最终验证 {row['final_validation']['loss']:.6f} | "
        f"实际更新 {row['actual_updates']}/{row['attempted_updates']} | "
        f"重载相同 {row['reload_state_equal']}"
    )
print("与 full 比较的最终参数最大误差:")
print(json.dumps(REPORT["comparisons"], ensure_ascii=False, indent=2))

参数量: 52464
full         | 初始验证 2.899294 → 最终验证 2.572053 | 实际更新 32/32 | 重载相同 True
accum        | 初始验证 2.899294 → 最终验证 2.572052 | 实际更新 32/32 | 重载相同 True
checkpoint   | 初始验证 2.899294 → 最终验证 2.572052 | 实际更新 32/32 | 重载相同 True
与 full 比较的最终参数最大误差:
{
  "accum": {
    "final_parameter_max_abs_vs_full": 1.3187527656555176e-06,
    "interpretation": "FP32/AMP 舍入可经多次 AdamW 更新放大；不能要求逐位相同。FP64 一步机制实验另查。"
  },
  "checkpoint": {
    "final_parameter_max_abs_vs_full": 1.3187527656555176e-06,
    "interpretation": "FP32/AMP 舍入可经多次 AdamW 更新放大；不能要求逐位相同。FP64 一步机制实验另查。"
  }
}


### 7.2 怎样解读结果？

我们关心的是同一个目标、同一组更新有没有因实现变化而被悄悄改变。合成数据很小，样本是符号重复模式；不把验证 loss 当作语言、编码或推理能力成绩。

这份讲义执行时，三路从约 `2.899294` 开始，最后约 `2.572053`，各完成 32 次更新。实际重跑时应以你的报告为准，特别是 AMP 与 CUDA 的舍入路径可能不同。

`full.pt`、`accum.pt`、`checkpoint.pt` 只有模型结构和参数等信息；不含优化器、随机数和数据进度，**不能精确恢复中断训练**。这里的激活 checkpoint 不等于训练恢复 checkpoint。

## 8. 在 3060 上测：要看一次完整更新的峰值，而非只看 forward

可选基准使用独立随机模型，固定有效批量 8、同一批输入和精度，对比 `8×1`、`2×4`、`2×4+checkpoint`。每次先做两次预热，让 Adam 状态分配完成；然后测完整 forward/backward/clip/step。计时不含输入 H2D 与 Python 数据整理。

预热后清空梯度到 None，CUDA 同步，记录 baseline allocated 并重置峰值；更新后同步，读取 peak allocated、增量和 peak reserved。各 trial 重新构建相同初态，顺序轮换，并保存原始样本。[峰值接口说明](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)

这是短模型诊断，不是严谨的硬件排行榜。小 micro 和重计算通常增加运行开销，具体时间应实测；CPU 字段不代表 GPU，缺失显存填 `null` 而不是 0。如果基准路径 OOM，会记录失败，不用成功子集伪装为完整测试。

In [24]:
benchmark_result = REPORT["resource_benchmark"]
if benchmark_result is None:
    print("本次未启用额外资源基准。顶部 RUN_RESOURCE_BENCHMARK=True 后重跑完整实验即可。")
else:
    print(json.dumps(benchmark_result["summary"], ensure_ascii=False, indent=2))
    print(benchmark_result["scope"])
    print(benchmark_result["limitations"])

本次未启用额外资源基准。顶部 RUN_RESOURCE_BENCHMARK=True 后重跑完整实验即可。


### 命令行运行方式

```bash
# 独立脚本默认 CPU、FP32，三路训练
python lesson_25_gradient_accumulation_and_checkpointing.py

# CPU 固定机制检查；不声称在 GPU 验证
python lesson_25_gradient_accumulation_and_checkpointing.py --checks-only

# 本机 CUDA/FP16；包含资源基准；先用短序列
python lesson_25_gradient_accumulation_and_checkpointing.py --device cuda --precision amp_fp16 --benchmark --bench-length 128

# GPU 预算不够放 full 路径时，只跑小 micro 的两条训练路径
python lesson_25_gradient_accumulation_and_checkpointing.py --device cuda --precision amp_fp16 --strategies accum,checkpoint

# CPU/BF16 的完整执行探测；支持与否由本机实际结果决定
python lesson_25_gradient_accumulation_and_checkpointing.py --precision amp_bf16 --epochs 1
```

`--strategies` 只控制主训练对照；独立的 `--benchmark` 自己仍会探测三条资源路径并记录 OOM。先确保某一条短序列训练可运行，再逐步增加长度。没有自动承诺你的 GPU 能容纳特定更大规模。

# 9. 独立练习：先停下来自己写

练习单元可以定义成功但不会默认执行。补全后把 `RUN_EX_*` 改为 `True`，通过测试再看答案。

## 练习 A｜一个 micro-batch 应贡献多少损失？

实现 `micro_loss_ex(logits, targets, window_tokens)`：返回仍可反向传播的标量；分子是忽略 `-100` 后的损失**和**，分母是整个窗口的有效目标数。

要求：`window_tokens` 是正整数，不能小于当前 micro 的有效目标数；当前 micro 至少包含一个有效目标；不得对 logits 先 softmax；不得使用 `.item()` 作为返回值。

In [25]:
def micro_loss_ex(logits, targets, window_tokens):
    # TODO: 检查分母与标签，计算 loss_sum / window_tokens。
    raise NotImplementedError("请完成练习 A")

In [26]:
def check_exercise_a(fn):
    logits = torch.tensor([[[1.0, 2.0, -1.0], [0.0, 1.0, 3.0]]], dtype=torch.float64, requires_grad=True)
    targets = torch.tensor([[1, IGNORE]])
    actual = fn(logits, targets, 5)
    expected = F.cross_entropy(logits[:, 0], torch.tensor([1]), reduction="sum") / 5
    torch.testing.assert_close(actual, expected)
    grad_actual, = torch.autograd.grad(actual, logits, retain_graph=True)
    grad_expected, = torch.autograd.grad(expected, logits)
    torch.testing.assert_close(grad_actual, grad_expected)
    assert torch.equal(grad_actual[:, 1], torch.zeros_like(grad_actual[:, 1]))
    for denom in (0, -1, True):
        try:
            fn(logits, targets, denom)
        except ValueError:
            pass
        else:
            raise AssertionError("非法 window_tokens 必须被拒绝")
    try:
        fn(logits, torch.tensor([[1, 2]]), 1)
    except ValueError:
        pass
    else:
        raise AssertionError("窗口分母不能小于 micro 有效目标数")
    try:
        fn(logits, torch.full_like(targets, IGNORE), 5)
    except ValueError:
        pass
    else:
        raise AssertionError("拒绝全忽略 micro")
    print("练习 A：数值、梯度、padding 与分母边界通过。")

RUN_EX_A = False
if RUN_EX_A:
    check_exercise_a(micro_loss_ex)
else:
    print("练习 A 尚未验收。")

练习 A 尚未验收。


## 练习 B｜一次 FP32 累积更新

实现 `accumulated_update_ex(model, optimizer, window, max_norm)`，只处理 FP32，不增加 AMP 分支。

步骤：`train()`；窗口开始清零；先统计有效 token 总数；按 micro 前向并立即反向；最后裁剪一次、更新一次；返回整个窗口的 token 平均 loss（Python 浮点数）。输入 batch 在 CPU 时按模型设备迁移。

拒绝空窗口、无有效目标的 micro、非正裁剪阈值。测试会比较**同一组初始参数的一次整批更新**，并统计 optimizer 是否只调用一次。

In [27]:
def accumulated_update_ex(model, optimizer, window, max_norm=1.0):
    # TODO: FP32 版本；禁止每个 micro 内清零、裁剪或 step。
    raise NotImplementedError("请完成练习 B")

In [28]:
def check_exercise_b(fn):
    cfg = Config(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32)
    rows = make_data()[0][:7]
    reference = new_model(cfg)
    student = copy.deepcopy(reference)
    opt_ref = torch.optim.SGD(reference.parameters(), lr=0.1)
    opt_student = torch.optim.SGD(student.parameters(), lr=0.1)
    x, y = collate(rows)
    expected = loss_sum(reference(x), y) / valid_tokens(y)
    opt_ref.zero_grad(set_to_none=True)
    expected.backward()
    nn.utils.clip_grad_norm_(reference.parameters(), 0.5)
    opt_ref.step()
    count = {"n": 0}
    def count_step(*_):
        count["n"] += 1
    hook = opt_student.register_step_post_hook(count_step)
    try:
        actual = fn(student, opt_student, list(iter_microbatches(rows, 3)), max_norm=0.5)
    finally:
        hook.remove()
    assert isinstance(actual, float) and math.isfinite(actual)
    assert count["n"] == 1, "一个窗口只能 step 一次"
    torch.testing.assert_close(torch.tensor(actual), expected.detach(), atol=2e-6, rtol=2e-6)
    for p, q in zip(student.parameters(), reference.parameters()):
        torch.testing.assert_close(p, q, atol=2e-6, rtol=2e-6)
    for bad in ([], [(x, torch.full_like(y, IGNORE))]):
        try:
            fn(student, opt_student, bad)
        except ValueError:
            pass
        else:
            raise AssertionError("非法窗口必须被拒绝")
    print("练习 B：token 加权、一次更新、全部参数与边界通过。")

RUN_EX_B = False
if RUN_EX_B:
    check_exercise_b(accumulated_update_ex)
else:
    print("练习 B 尚未验收。")

练习 B 尚未验收。


## 练习 C｜保持随机行为的逐块重计算

实现 `checkpoint_blocks_ex(blocks, x, allowed)`。逐个调用传入的 Block，并启用非重入检查点和 RNG 保留；返回最后的隐藏状态。

不使用闭包读取后来改变的循环变量，不对 x 做 `detach()`，不使用 `no_grad()`。测试打开 Dropout，核对输出、输入梯度和所有块参数的梯度，避免“前向相同就算通过”的误判。

In [29]:
def checkpoint_blocks_ex(blocks, x, allowed):
    # TODO: checkpoint(block, x, allowed, use_reentrant=False, preserve_rng_state=True)
    raise NotImplementedError("请完成练习 C")

In [30]:
def check_exercise_c(fn):
    cfg = Config(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32, dropout=0.25)
    original = new_model(cfg).blocks
    student = copy.deepcopy(original)
    x0 = torch.randn(2, 4, 16, generator=torch.Generator().manual_seed(2550))
    mask = torch.ones(4, 4, dtype=torch.bool).tril()[None, None]
    outputs, inputs = [], []
    for blocks, use_student in [(original, False), (student, True)]:
        x = x0.clone().requires_grad_()
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(2551)
            if use_student:
                out = fn(blocks, x, mask)
            else:
                out = x
                for block in blocks:
                    out = block(out, mask)
            out.square().mean().backward()
        outputs.append(out.detach())
        inputs.append(x.grad.clone())
    torch.testing.assert_close(outputs[0], outputs[1], atol=1e-6, rtol=1e-5)
    torch.testing.assert_close(inputs[0], inputs[1], atol=1e-6, rtol=1e-5)
    for p, q in zip(original.parameters(), student.parameters()):
        assert p.grad is not None and q.grad is not None
        torch.testing.assert_close(p.grad, q.grad, atol=1e-6, rtol=1e-5)
    print("练习 C：Dropout 前向、输入梯度与所有块梯度通过。")

RUN_EX_C = False
if RUN_EX_C:
    check_exercise_c(checkpoint_blocks_ex)
else:
    print("练习 C 尚未验收。")

练习 C 尚未验收。


---
# 10. 参考答案

下面的函数使用 `_answer` 后缀，不覆盖你的实现。完成后再运行核对。答案只是一种可行实现，必须理解分母、反传和参数更新的作用域，而不是仅复制函数名。

In [31]:
def micro_loss_answer(logits, targets, window_tokens):
    positive_int(window_tokens, "window_tokens")
    n = valid_tokens(targets)
    if n <= 0 or window_tokens < n:
        raise ValueError("micro 无目标，或窗口分母小于 micro 目标数")
    return loss_sum(logits, targets) / window_tokens

check_exercise_a(micro_loss_answer)

练习 A：数值、梯度、padding 与分母边界通过。


In [32]:
def accumulated_update_answer(model, optimizer, window, max_norm=1.0):
    if not window or not math.isfinite(max_norm) or max_norm <= 0:
        raise ValueError("窗口和裁剪阈值不合法")
    counts = [valid_tokens(y) for _, y in window]
    if any(n == 0 for n in counts):
        raise ValueError("micro 中没有有效目标")
    total_tokens = sum(counts)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    device = next(model.parameters()).device
    total_loss = 0.0
    for x, y in window:
        x, y = x.to(device), y.to(device)
        summed = loss_sum(model(x), y)
        (summed / total_tokens).backward()
        total_loss += float(summed.detach())
        del summed
    nn.utils.clip_grad_norm_(model.parameters(), max_norm, error_if_nonfinite=True)
    optimizer.step()
    return total_loss / total_tokens

check_exercise_b(accumulated_update_answer)

练习 B：token 加权、一次更新、全部参数与边界通过。


In [33]:
def checkpoint_blocks_answer(blocks, x, allowed):
    for block in blocks:
        x = checkpoint(block, x, allowed, use_reentrant=False, preserve_rng_state=True)
    return x

check_exercise_c(checkpoint_blocks_answer)

练习 C：Dropout 前向、输入梯度与所有块梯度通过。


## 11. 常见故障的定位顺序

| 现象 | 优先检查 |
|---|---|
| 累积后 loss 明显变了 | 是否错误平均 micro 均值、是否遗漏尾窗口、是否先 softmax |
| 每轮更新次数太多 | `step()` 是否写在 micro 循环内；调度器是否同样写错 |
| 显存没怎么下降 | 是否把带图 loss/logits 保存到列表，是否最后统一 backward |
| 开启检查点后梯度不同 | Dropout RNG、闭包绑定、共享可变状态、`detach()`、mode |
| loss 有限但参数不动 | 检查 GradScaler 是否因非有限梯度跳过更新 |
| 参数本身放不下 | 累积与检查点主要减少激活，不能直接缩小全部参数/Adam 状态 |
| CPU 资源报告有 `null` | 没有 GPU allocator 数据，不是“占用为零” |

不使用 `retain_graph=True` 来掩盖重复反传错误；不使用 `torch.no_grad()` 来掩盖检查点实现问题；不要忽略 warning 或异常后继续报告“成功”。

**验收标准**：能推导 token 分母；解释 43 条数据为何产生 4 次更新；写对 AMP 的窗口边界；解释前向一样但梯度不同的 Dropout 反例；准确说明一个内存数字统计了什么、没有统计什么。

## 12. 参考资料与下一课

- [PyTorch 2.10：activation checkpoint](https://docs.pytorch.org/docs/2.10/checkpoint.html)：非重入、RNG、重算状态及限制。
- [AMP examples：梯度累积](https://docs.pytorch.org/docs/2.10/notes/amp_examples.html)：窗口内固定 scale，窗口末去缩放、裁剪、更新。
- [PyTorch 官方：激活检查点的速度/内存取舍](https://pytorch.org/blog/activation-checkpointing-techniques/)：为何减少保存会增加重算。
- [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)：保存张量与 hooks。
- [CUDA max_memory_allocated](https://docs.pytorch.org/docs/stable/generated/torch.cuda.memory.max_memory_allocated.html)：allocator 峰值口径。

代码在 PyTorch 2.10.0+cpu、Python 3.13 环境验证；链接中 stable/main 页面可能继续变化，不声称所有历史版本都行为一致。CPU 实测不代表 RTX 3060 结果。默认代码未使用 `torch.compile`、DDP、ZeRO、offload 或 KV cache。

**下一课：第 26 课——训练检查点与断点续训。** 保存的不只是模型权重，还包括优化器、scaler、随机数与数据进度；用“连续训练”和“中断后恢复”的对照检查恢复是否正确。